# T02 — Q, K, V y el producto escalar escalado

## 1. Título y paper

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017)  
**Fuente primaria:** [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
**Foco de esta miniatura:** la ecuación 1 del paper  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)


## 2. Objetivos

1. Explicar qué papel juega cada uno de Q, K y V.
2. Implementar `softmax(QKᵀ/√d_k)·V` desde cero y verificar sus propiedades.


## 3. Prerrequisitos

- Python 3.11+ con el paquete instalado (`pip install -e .`).
- Notebook [`P08_transformer`](P08_transformer.ipynb) al menos hojeado.
- Álgebra de vectores: producto escalar, norma y softmax.


## 4. Intuición

Una búsqueda en una biblioteca: **Q** es lo que preguntas, **K** son las etiquetas de los lomos con las que comparas, y **V** es el contenido que te llevas. Comparas contra K, pero te llevas V.


## 5. Concepto mínimo

```text
Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V
```

`QKᵀ` mide compatibilidad; `√d_k` normaliza la escala; `softmax` convierte en pesos que suman 1; multiplicar por `V` mezcla la información.


## 6. Código explicado

Código mínimo, sin dependencias externas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
from ai_evolution.papers_lab import scaled_dot_product_attention

Q = [[1.0, 0.0, 0.0, 0.0]]
K = [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.9, 0.1, 0.0, 0.0]]
V = [[10.0, 0.0], [0.0, 10.0], [5.0, 5.0]]
r = scaled_dot_product_attention(Q, K, V)
print('pesos :', [round(w, 4) for w in r['weights'][0]], '· suma:', round(sum(r['weights'][0]), 6))
print('salida:', [round(v, 4) for v in r['output'][0]])

## 7. Predicción antes de ejecutar

Q coincide con K[0] y se parece a K[2]. ¿Cuál de los tres valores dominará la salida?

> Escribe tu respuesta antes de continuar.


## 8. Experimento controlado


In [ ]:
for etiqueta, escala in (('con √d_k', True), ('sin escala', False)):
    r = scaled_dot_product_attention(Q, K, V, scale=escala)
    print(f"{etiqueta:<11} pesos={[round(w, 4) for w in r['weights'][0]]}")

## 9. Salida interpretable

Los pesos suman exactamente 1 y la salida es una **combinación convexa** de las filas de V: nunca puede salirse del casco convexo de los valores. La atención mezcla, no inventa.


## 10. Comentario pedagógico

Esta miniatura aísla **una** pieza del bloque. Aislar es didáctico y también es una simplificación: en el modelo real todas las piezas interactúan y se entrenan juntas.


## 11. Error o anti-patrón deliberado


In [ ]:
malos = [2.0, -1.0, 0.5]
s = sum(malos)
print('normalizar dividiendo por la suma:', [round(m / s, 3) for m in malos])
print('→ hay pesos negativos y > 1: ya no es una distribución de probabilidad')

## 12. Corrección


In [ ]:
import math
e = [math.exp(m) for m in malos]
print('softmax:', [round(v / sum(e), 4) for v in e], '· suma:', round(sum(v / sum(e) for v in e), 6))

## 13. Desafío guiado

Haz Q ortogonal a todas las filas de K. ¿Qué distribución sale y qué significa esa entropía máxima?


## 14. Desafío autónomo

Reescribe esta pieza con proyecciones aprendidas y comprueba que tu implementación reproduce las propiedades verificadas aquí (sumas, formas, invariantes). Documenta la semilla.


## 15. Evidencia de aprendizaje

Guarda la salida del experimento, tu predicción previa y una frase sobre qué invariante acabas de verificar.


## 16. Cierre

Pieza cubierta: **la ecuación 1 del paper**. Ya puede describirse con precisión, sin metáforas.


## 17. Conexión con el siguiente hito

El softmax es la pieza que convierte compatibilidad en distribución. Vale la pena mirarlo solo (T03).
